In [1]:
import os
import sys
from autogen_agentchat.agents import AssistantAgent
from autogen_core.memory import MemoryContent, MemoryMimeType
from autogen_ext.memory.chromadb import (ChromaDBVectorMemory,
    PersistentChromaDBVectorMemoryConfig)
from chromadb.utils import embedding_functions
from sentence_transformers import SentenceTransformer
from autogen_agentchat.ui import Console

# Project import paths
# Get the current directory (which is TFM_BA_GPT/test)
notebook_dir = os.getcwd()

# Compute absolute paths for lib and src (one level up from test/)
lib_path = os.path.abspath(os.path.join(notebook_dir, '..', 'lib'))
src_path = os.path.abspath(os.path.join(notebook_dir, '..', 'src'))

sys.path.append(lib_path)
sys.path.append(src_path)

from model_loader import load_model_client

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = SentenceTransformer('all-MiniLM-L6-v2')
model = model.to('cuda')  # This moves the model to GPU

# embedding_function = (
#         embedding_functions.SentenceTransformerEmbeddingFunction(
#         model_name="paraphrase-multilingual-mpnet-base-v2"
#         )
#     )



# Using a specific Sentence Transformer model
memory = ChromaDBVectorMemory(
    config=PersistentChromaDBVectorMemoryConfig(
        collection_name="experiences",
        persistence_path="/run/media/jun/Jun_6T/Code/TFM_BA_GPT/.experiences",
        k=3,
        score_threshold=0.4,
        embedding_function=model,
    )
    )

In [3]:
# Create a model client
model_client = load_model_client("azure")
# Create a memory instance


agent = AssistantAgent(
    name="assistant",
    model_client=model_client,
    memory=[memory],
    system_message="You are a helpful assistant that remembers user preferences.",
)
 # Add something to memory
#test_content = "path for process_diagram.mmd is in folder /run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.mmd"
# test_content = {
#     "functional_specification": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/functional_specification.formatted.md",
#     "interview_data": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/interview.jsonl",
#     "process_diagram_source": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.mmd",
#     "process_diagram_image": "/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.png"
# }

test_content = """
## 📁 File Folders

### Session: `sess-20231011162620`

    **Functional Specification**  
    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/functional_specification.formatted.md`

    **Interview Data**  
    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/interview.jsonl`

    **Process Diagram (Source)**  
    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.mmd`

    **Process Diagram (Image)**  
    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.png`
"""

await memory.add(MemoryContent(content=test_content, mime_type=MemoryMimeType.MARKDOWN,))
# Create an assistant agent with memory
# Run a sample task
result = await Console(
    agent.run_stream(task="In what foler I can find process_diagram.mmd"),
    output_stats=True,  # Enable stats printing.
)
print(result.messages[-1].content)  # type: ignore


---------- TextMessage (user) ----------
In what foler I can find process_diagram.mmd
---------- MemoryQueryEvent (assistant) ----------
[MemoryContent(content='\n## 📁 File Folders\n\n### Session: `sess-20231011162620`\n\n    **Functional Specification**  \n    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/functional_specification.formatted.md`\n\n    **Interview Data**  \n    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/interview.jsonl`\n\n    **Process Diagram (Source)**  \n    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.mmd`\n\n    **Process Diagram (Image)**  \n    `/run/media/jun/Jun_6T/Code/TFM_BA_GPT/experience/sess-20231011162620/process_diagram.png`\n', mime_type='MemoryMimeType.MARKDOWN', metadata={'mime_type': 'MemoryMimeType.MARKDOWN', 'score': 0.4111160635948181, 'id': 'cd240681-e303-423c-bcab-9e744507969d'}), MemoryContent(content='\n## 📁 File Folders\n\n### Session: `sess-202310111

/run/media/jun/Jun_6T/Code/TFM_BA_GPT/venv/lib/python3.13/site-packages/autogen_agentchat/agents/_assistant_agent.py:1107: UserWarning: Resolved model mismatch: gpt-4o-2024-08-06 != gpt-4o-2024-11-20. Model mapping in autogen_ext.models.openai may be incorrect. Set the model to gpt-4o-2024-11-20 to enhance token/cost estimation and suppress this warning.
  model_result = await model_client.create(
